# Dataset Loading

In [1]:
from tensordict import TensorDict
from torch.utils.data import DataLoader

from main.core_data.ds.td_dataset import TdSegmentedExperimentDataset

ds_path = "/home/jacopo/dataset/EEGAVI/FACED-PROBE/interleaved"
spec_path = "/home/jacopo/dataset/EEGAVI/FACED-PROBE/interleaved/spec.csv"
# All experiments have a fixed 30s window, so I don't have to pad, it wil work out of the box.

ds = TdSegmentedExperimentDataset(
    ds_path, spec_path, accessible_user_ids=[80, 79, 78]
)


def td_collate(batch):
    return TensorDict.stack([b.exclude("meta") for b in batch], dim=0)


dl = DataLoader(ds, shuffle=True, batch_size=4, collate_fn=td_collate)
next(iter(dl))

TensorDict(
    fields={
        assessment: TensorDict(
            fields={
                labels: NonTensorStack(
                    [[['joy', 'tenderness', 'inspiration', 'amusement'...,
                    batch_size=torch.Size([4, 3, 12]),
                    device=cpu),
                scales: NonTensorStack(
                    [[[[[0.0, 7.0], [0.0, 7.0], [0.0, 7.0], [0.0, 7.0]...,
                    batch_size=torch.Size([4, 3, 1, 12, 2]),
                    device=cpu),
                scores: Tensor(shape=torch.Size([4, 3, 12]), device=cpu, dtype=torch.float64, is_shared=False)},
            batch_size=torch.Size([4, 3]),
            device=cpu,
            is_shared=False),
        eeg: TensorDict(
            fields={
                data: Tensor(shape=torch.Size([4, 3, 32, 19, 200]), device=cpu, dtype=torch.int8, is_shared=False),
                mask: Tensor(shape=torch.Size([4, 3, 32, 19]), device=cpu, dtype=torch.bool, is_shared=False),
                scales: Ten

# Model creation

In [2]:
import torch

weights_path = "/home/jacopo/PycharmProjects/progetto-tesi/epochepoch=45-stepstep=117484.ckpt"
ckpt = torch.load(weights_path, map_location="cpu")
my_ckpt = dict()
for key, value in ckpt["state_dict"].items():
    if key.startswith("student."):
        my_ckpt[key[8:]] = value

my_ckpt.keys()

dict_keys(['cls_token', 'pivot.adapter.ff.0.weight', 'pivot.adapter.ff.0.bias', 'pivot.adapter.ff.2.weight', 'pivot.adapter.ff.2.bias', 'supports.0.adapter.linear_reshape.weight', 'supports.0.adapter.linear_reshape.bias', 'supports.0.adapter.resampler.latents', 'supports.0.adapter.resampler.blocks.0.0.kv_gate', 'supports.0.adapter.resampler.blocks.0.0.norm_latents.weight', 'supports.0.adapter.resampler.blocks.0.0.norm_latents.bias', 'supports.0.adapter.resampler.blocks.0.0.norm_media.weight', 'supports.0.adapter.resampler.blocks.0.0.norm_media.bias', 'supports.0.adapter.resampler.blocks.0.0.to_q.weight', 'supports.0.adapter.resampler.blocks.0.0.to_k.weight', 'supports.0.adapter.resampler.blocks.0.0.to_v.weight', 'supports.0.adapter.resampler.blocks.0.0.to_out.weight', 'supports.0.adapter.resampler.blocks.0.1.net.0.weight', 'supports.0.adapter.resampler.blocks.0.1.net.0.bias', 'supports.0.adapter.resampler.blocks.0.1.net.1.weight', 'supports.0.adapter.resampler.blocks.0.1.net.3.weight',

In [3]:
from main.model.downstream.faced.model import FacedLinearProbe
from main.model.downstream.faced.training import FacedProbeTrainer
import lightning
from main.model.neegavi.factory import Factory

backbone = Factory.best_inference(
    "/home/jacopo/PycharmProjects/progetto-tesi/epochepoch=45-stepstep=117484.ckpt",
)

backbone.load_state_dict(my_ckpt, strict=False)
backbone.eval()

model = FacedLinearProbe(backbone=backbone, in_dim=384, out_dim=12)
trainable_model = FacedProbeTrainer(model=model)
trainable_model = FacedProbeTrainer(model=model)
trainer = lightning.Trainer(accelerator="cpu", devices=1, max_epochs=1, max_steps=2, precision="16-mixed", )

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/home/jacopo/PycharmProjects/progetto-tesi/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/accelerator_connector.py:508: You passed `Trainer(accelerator='cpu', precision='16-mixed')` but AMP with fp16 is not supported on CPU. Using `precision='bf16-mixed'` instead.
Using bfloat16 Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/jacopo/PycharmProjects/progetto-tesi/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.


In [4]:
res = trainer.fit(trainable_model, dl)

/home/jacopo/PycharmProjects/progetto-tesi/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
/home/jacopo/PycharmProjects/progetto-tesi/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:231: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name         | Type             | Params | Mode 
----------------------------------------------------------
0 | model        | FacedLinearProbe | 33.1 M | train
1 | val_pearson  | PearsonCorrCoef  | 0      | train
2 | test_pearson | PearsonCorrCoef  | 0      | train
----------------------------------------------------------
33.1 M    Trainable params
0         Non-trainable params
33.1 M    Total params
132.526   Total estimated model params size (MB)
4         Modules in train mode
221       Module

Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/home/jacopo/PycharmProjects/progetto-tesi/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3678: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
